# Aligned Summary — LLM-Judge Prompt Optimiser

Iteratively aligns a judge prompt with human-expert golden labels from
`summary_golden_test_cases.xlsx`.

**Flow**
1. Start with `prompt_0` — a judge that evaluates AI-generated debt advisory summary fields.
2. For each round, send every test-case row to the webhook with `{"input": filled_prompt}`.
3. Parse the JSON response; compare predicted labels against golden labels.
4. If **all** fields have mismatch rate ≤ 30 % → stop.
5. Otherwise feed mismatched rows into `OptPrompt` to obtain an improved prompt, then repeat.
6. Stop after **5 rounds** at most.

In [13]:
import os
import re
import json
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
N8N_BASE_URL       = "https://alphamakeathon-automation.arisetech.dev"
WEBHOOK_PATH       = "9d6049fc-77c8-43e9-b71e-f734506f4f9d"   # ← fill in
USE_TEST_URL       = False   # True → /webhook-test/…  |  False → /webhook/…

GOLDEN_FILE        = "golden_test_cases/summary_golden_test_cases.xlsx"
RESULT_DIR         = "summary_test_result"

MAX_ROUNDS         = 5
MISMATCH_THRESHOLD = 0.30    # stop when ALL fields are within this rate
TIMEOUT            = 600      # seconds per request (10 minutes)
RETRIES            = 2
DELAY_BETWEEN      = 0.5     # seconds between rows

SCORE_FIELDS = ["subject", "objective", "debt_cause", "offer_suitability", "request_information", "summary"]


def golden_col(f: str) -> str:
    """Map a score-field name to its golden-label column in the DataFrame."""
    return f"rating_{f}"


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


print("Webhook URL:", get_webhook_url())

Webhook URL: https://alphamakeathon-automation.arisetech.dev/webhook/9d6049fc-77c8-43e9-b71e-f734506f4f9d


## OptPrompt

Meta-prompt sent to the webhook when mismatches exceed the threshold.
Placeholders: **`{oldPrompt}`**, **`{mismatchResults}`**.
The LLM must return an *UpdatePrompt* whose own placeholders cover all customer-context
fields and generated summary fields listed below.

In [14]:
OptPrompt = (
    "You are an expert prompt engineer specialising in LLM-as-a-judge systems "
    "for evaluating AI-generated debt advisory summaries at a Thai commercial bank.\n\n"
    "Your task is to improve the judge prompt below so it better aligns with "
    "human-expert (golden) evaluations.\n\n"
    "## Current Prompt\n"
    "{oldPrompt}\n\n"
    "## Mismatch Cases\n"
    "The following rows show where the current prompt disagreed with golden labels. "
    "Each entry includes the customer context, generated summary fields, "
    "the golden label, and the predicted label:\n"
    "{mismatchResults}\n\n"
    "## Your Task\n"
    "1. Analyse the mismatch patterns.\n"
    "2. Identify what the current prompt misinterprets.\n"
    "3. Rewrite the prompt to fix those issues while preserving correct behaviours.\n\n"
    "## Requirements for the output prompt\n"
    "- MUST keep exactly these placeholders (with single curly braces):\n"
    "  Customer context: {userMessage}, {LastAImessage}, {userMessageSummary}, {narrative},\n"
    "  {preference}, {maxPayment}, {ageRange}, {employmentType}, {monthsAsCustomer},\n"
    "  {dpd}, {sumOsNCB}, {installmentNCB_Y1}, {installmentNCB_Y2}, {installmentNCB_Y3}, {offerReadableText}\n"
    "  Generated fields: {result_subject}, {result_objective}, {result_debt_cause},\n"
    "  {result_offer_suitability}, {result_request_information}, {result_summary}\n"
    "- MUST instruct the LLM to return ONLY a JSON object with these fields and allowed values:\n"
    '    { "subject":              "good"|"acceptable"|"invalid",\n'
    '      "objective":           "good"|"acceptable"|"invalid",\n'
    '      "debt_cause":          "good"|"acceptable"|"invalid",\n'
    '      "offer_suitability":   "good"|"acceptable"|"invalid",\n'
    '      "request_information": "good"|"acceptable"|"invalid",\n'
    '      "summary":             "good"|"acceptable"|"invalid" }\n\n'
    "Return ONLY the improved prompt text — no preamble, no explanation."
)

print("OptPrompt defined:", len(OptPrompt), "chars")

OptPrompt defined: 1750 chars


## Initial Judge Prompt — `prompt_0`

Evaluates the quality of each field in an AI-generated debt advisory summary.
Placeholders: all customer-context fields plus **`{result_subject}`**, **`{result_objective}`**,
**`{result_debt_cause}`**, **`{result_offer_suitability}`**, **`{result_request_information}`**,
**`{result_summary}`**.
Returns JSON with `subject`, `objective`, `debt_cause`, `offer_suitability`,
`request_information`, `summary`, each valued **`"good"`** / **`"acceptable"`** / **`"invalid"`**.

In [15]:
prompt_0 = (
    "You are a Quality Assurance (QA) expert evaluating AI-generated debt advisory summaries "
    "for bank staff at a Thai commercial bank.\n\n"
    "The AI assistant generates structured summaries based on customer debt situations and "
    "recommended solutions. Your task is to evaluate whether each generated field is "
    '"good" (accurate and meets quality standards), '
    '"acceptable" (mostly correct with minor issues), or '
    '"invalid" (inaccurate, inappropriate, or fails quality standards).\n\n'
    "## Customer Context\n\n"
    "Customer's last message:\n{userMessage}\n\n"
    "Last AI message (offer/recommendation):\n{LastAImessage}\n\n"
    "All customer messages (chronological):\n{userMessageSummary}\n\n"
    "Conversation narrative:\n{narrative}\n\n"
    "## Financial Profile\n"
    "Financial preference: {preference}\n"
    "Max affordable monthly payment: {maxPayment} THB\n"
    "Customer age range: {ageRange}\n"
    "Employment type: {employmentType}\n"
    "Months as bank customer: {monthsAsCustomer}\n"
    "Days past due (DPD): {dpd}\n"
    "Total outstanding debt (NCB): {sumOsNCB} THB\n"
    "NCB installments — Y1: {installmentNCB_Y1} THB/mo | Y2: {installmentNCB_Y2} THB/mo | Y3: {installmentNCB_Y3} THB/mo\n\n"
    "## Recommended Offer\n"
    "{offerReadableText}\n\n"
    "## Generated Summary to Evaluate\n"
    "subject: {result_subject}\n"
    "objective: {result_objective}\n"
    "debt_cause: {result_debt_cause}\n"
    "offer_suitability: {result_offer_suitability}\n"
    "request_information: {result_request_information}\n"
    "summary: {result_summary}\n\n"
    "## Evaluation Criteria\n\n"
    "### subject (หัวข้อทางการ)\n"
    '- "good": Formal Thai bank memo title; clearly states the type of request '
    "(e.g. ขอปรับโครงสร้างหนี้, ขอรวมหนี้และเปิดสินเชื่อใหม่, ขอขยายระยะเวลาผ่อนชำระ); "
    "concise ≤ 15 words; no customer name or account numbers.\n"
    '- "acceptable": Mostly correct formal title; request type is identifiable but slightly '
    "imprecise in wording; still usable as a memo header without misleading staff.\n"
    '- "invalid": Not formal, too vague to be actionable, includes customer identifiers, '
    "misstates the request type, or describes an action that does not match the offer.\n\n"
    "### objective (วัตถุประสงค์)\n"
    '- "good": 2–4 sentences accurately describing what staff must process; correctly '
    "identifies the offer type (TDR, debt consolidation + new loan opening, term extension, etc.); "
    "if consolidation is involved, explicitly states new loan opening; "
    "mentions interest rate improvement only when the new rate is explicitly lower than before.\n"
    '- "acceptable": Correct action identified with minor imprecision; '
    "overall intent is clear and staff could proceed, but one detail is slightly off "
    "(e.g. slightly vague on offer type, minor sentence-count deviation).\n"
    '- "invalid": Wrong action identified, incorrectly states new loan opening when not applicable, '
    "mentions interest rate when not lower or not relevant, or significantly misrepresents the offer.\n\n"
    "### debt_cause (สาเหตุหนี้)\n"
    '- "good": 2–4 sentences summarising root causes based on the conversation narrative; '
    "covers relevant aspects (financial burden, income instability, external factors); "
    "does not speculate beyond what the customer stated; empathetic but objective tone.\n"
    '- "acceptable": Captures the main debt cause but misses secondary factors or is slightly '
    "general; no speculation or contradiction; still useful for staff understanding.\n"
    '- "invalid": Speculates beyond stated facts, contradicts the narrative, '
    "ignores all key causes mentioned, or is so vague as to be uninformative.\n\n"
    "### offer_suitability (เหตุผลสนับสนุน)\n"
    '- "good":\n'
    "  CASE 1 (offer accepted — userMessage indicates the customer selected an offer): "
    "Explains why the specific offer suits the customer, referencing relevant factors "
    "(installment reduction, alignment with maxPayment, employment type compatibility, "
    "term extension relative to financial capacity). Does not mention rate improvement unless explicitly lower.\n"
    "  CASE 2 (no offer accepted — customer made a different request or asked a question): "
    "Provides objective evidence supporting or not supporting the request, "
    "citing relevant factors (DPD, NCB burden, MOB, stated hardship). Professional and neutral.\n"
    "  Both cases: 2–4 sentences maximum.\n"
    '- "acceptable": Correct case applied; main reasoning is sound but one relevant factor '
    "is missing or minor wording is imprecise; staff can still use it to evaluate the case.\n"
    '- "invalid": Wrong case applied, irrelevant factors cited, unsupported claims, '
    "incorrect rate reference, or reasoning contradicts the customer data.\n\n"
    "### request_information (ข้อมูลเพิ่มเติม)\n"
    '- "good": Accurately extracts all specific customer requests, conditions, preferred contact '
    "times, or follow-up items explicitly stated in the conversation; "
    "uses \"ไม่มีข้อมูลเพิ่มเติม\" only when the customer truly mentioned nothing additional.\n"
    '- "acceptable": Captures the main request information but misses a minor detail; '
    "no fabrication; still useful for staff follow-up.\n"
    '- "invalid": Misses major explicit customer requests, fabricates information not stated '
    "by the customer, or uses \"ไม่มีข้อมูลเพิ่มเติม\" when the customer did state specific requests.\n\n"
    "### summary (สรุปภาพรวม)\n"
    '- "good": 3–5 sentence Thai paragraph covering all of: '
    "who the customer is (employment, financial standing), the core debt problem, "
    "what is being requested or offered, and why it is appropriate or what staff should consider; "
    "professional neutral tone; synthesises content rather than repeating verbatim from other fields.\n"
    '- "acceptable": Covers most required elements (missing at most one); '
    "professional tone; minor verbatim overlap but still readable as a standalone briefing.\n"
    '- "invalid": Missing two or more key elements, unprofessional tone, '
    "excessive verbatim repetition from other fields, or outside the 3–5 sentence range.\n\n"
    "## Output Instructions\n"
    "Return ONLY a JSON object — no markdown, no explanation:\n"
    "{\n"
    '  "subject":              "good" | "acceptable" | "invalid",\n'
    '  "objective":           "good" | "acceptable" | "invalid",\n'
    '  "debt_cause":          "good" | "acceptable" | "invalid",\n'
    '  "offer_suitability":   "good" | "acceptable" | "invalid",\n'
    '  "request_information": "good" | "acceptable" | "invalid",\n'
    '  "summary":             "good" | "acceptable" | "invalid"\n'
    "}"
)

print("prompt_0 defined:", len(prompt_0), "chars")

prompt_0 defined: 6035 chars


## Helper Functions

In [16]:
_PROMPT_FIELDS = [
    "userMessage", "LastAImessage", "userMessageSummary", "narrative",
    "preference", "maxPayment", "ageRange", "employmentType", "monthsAsCustomer",
    "dpd", "sumOsNCB", "installmentNCB_Y1", "installmentNCB_Y2", "installmentNCB_Y3",
    "offerReadableText",
    "result_subject", "result_objective", "result_debt_cause",
    "result_offer_suitability", "result_request_information", "result_summary",
]


def fill_prompt(template: str, row: pd.Series) -> str:
    filled = template
    for col in _PROMPT_FIELDS:
        filled = filled.replace(f"{{{col}}}", str(row.get(col, "")))
    return filled


def fill_opt_prompt(template: str, oldPrompt: str, mismatchResults: str) -> str:
    return (template
            .replace("{oldPrompt}",       str(oldPrompt))
            .replace("{mismatchResults}", str(mismatchResults)))


def _call_raw(payload: dict, timeout: int = TIMEOUT, retries: int = RETRIES):
    url = get_webhook_url()
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last_exc


def call_webhook(prompt: str):
    return _call_raw({"input": prompt})


def get_response_text(resp) -> str:
    if isinstance(resp, str):
        return resp
    if isinstance(resp, dict):
        for key in ("output", "text", "result", "response", "content", "message"):
            if key in resp and resp[key] is not None:
                return str(resp[key])
        return json.dumps(resp, ensure_ascii=False)
    return str(resp)


def parse_json_response(resp) -> dict:
    text = get_response_text(resp)
    # 1. markdown code block
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # 2. whole text as JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # 3. first JSON object found
    m = re.search(r"(\{[\s\S]*?\})", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    return {}


def extract_prompt_text(resp) -> str:
    text = get_response_text(resp)
    m = re.search(r"```(?:\w+)?\s*([\s\S]*?)\s*```", text)
    if m:
        return m.group(1).strip()
    return text.strip()


print("Helpers ready.")

Helpers ready.


## Load Golden Test Cases

In [17]:
df_golden = pd.read_excel(GOLDEN_FILE)

# Normalise labels to lowercase so they match the judge's output values
for f in SCORE_FIELDS:
    df_golden[golden_col(f)] = df_golden[golden_col(f)].str.strip().str.lower()

print(f"Loaded {len(df_golden)} golden test cases")
print("Columns:", df_golden.columns.tolist())
print("\nLabel distributions:")
for f in SCORE_FIELDS:
    print(f"  {golden_col(f)}: {df_golden[golden_col(f)].value_counts().to_dict()}")
df_golden.head(3)

Loaded 30 golden test cases
Columns: ['testId', 'inputPath', 'userMessage', 'LastAImessage', 'userMessageSummary', 'narrative', 'preference', 'maxPayment', 'ageRange', 'employmentType', 'monthsAsCustomer', 'dpd', 'sumOsNCB', 'installmentNCB_Y1', 'installmentNCB_Y2', 'installmentNCB_Y3', 'offerReadableText', 'situation_tested', 'result_subject', 'result_objective', 'result_debt_cause', 'result_offer_suitability', 'result_request_information', 'result_summary', 'error', 'rating_subject', 'rating_objective', 'rating_debt_cause', 'rating_offer_suitability', 'rating_request_information', 'rating_summary', 'reason_subject', 'reason_objective', 'reason_debt_cause', 'reason_offer_suitability', 'reason_request_information', 'reason_summary']

Label distributions:
  rating_subject: {'good': 22, 'invalid': 4, 'acceptable': 4}
  rating_objective: {'good': 22, 'acceptable': 5, 'invalid': 3}
  rating_debt_cause: {'good': 21, 'acceptable': 6, 'invalid': 3}
  rating_offer_suitability: {'good': 22, 'ac

,testId,inputPath,userMessage,LastAImessage,userMessageSummary,narrative,preference,maxPayment,ageRange,employmentType,...,rating_debt_cause,rating_offer_suitability,rating_request_information,rating_summary,reason_subject,reason_objective,reason_debt_cause,reason_offer_suitability,reason_request_information,reason_summary
0,BAD-TC-0002,webhook,The user select one of the offer provided by AI,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,message 1: สนใจแผนที่เสนอครับ,"The customer, aiming for a debt restructuring ...",NaN,Not provided,55-59 years old,อาชีพอิสระ/เกษตรกร (Self-Employed/Farmer),...,good,good,good,invalid,Invalid: wrong request type or contradicts the...,Invalid: requests a different action/product f...,Relaxed review: usable explanation of debt pre...,Relaxed review: correct acceptance/request cas...,Relaxed review: factual and sufficient; no exp...,Invalid: fabricates major facts or contradicts...
1,BAD-TC-0017,webhook,The user select one of the offer provided by AI,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,message 1: สนใจแผนที่เสนอครับ,The customer is seeking debt restructuring and...,NaN,Not provided,55-59 years old,อาชีพอิสระ/เกษตรกร (Self-Employed/Farmer),...,invalid,good,invalid,acceptable,Invalid: wrong request type or contradicts the...,Relaxed review: actionable and aligned with th...,Invalid: fabricates major customer circumstanc...,Relaxed review: correct acceptance/request cas...,Invalid: fabricates a customer request or cont...,"Acceptable: usable briefing, but profile/raw-v..."
2,BAD-TC-0006,webhook,The user select one of the offer provided by AI,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,message 1: สนใจแผนที่เสนอครับ,The customer has reviewed the debt restructuri...,"{""DebtSituation"":""DebtBurden""}",800 THB,55-59 years old,พนักงานประจำ (Salaried),...,good,invalid,good,good,Invalid: wrong request type or contradicts the...,"Acceptable: action is broadly correct, but the...",Relaxed review: usable explanation of debt pre...,Invalid: uses the wrong CASE or contradicts wh...,Relaxed review: factual and sufficient; no exp...,Relaxed review: coherent staff briefing coveri...


## Optimisation Loop

For each round:
1. Fill all customer-context and generated-summary placeholders in the current judge prompt.
2. POST `{"input": filled_prompt}` to the webhook; collect responses.
3. Parse each response string → dict with `re` / `json`.
4. Save `test_result_prompt_N.xlsx`.
5. Compute per-field mismatch rate vs golden labels (`rating_*` columns, normalised to lowercase).
6. **Stop** if every field's rate ≤ 30 %, or after 5 rounds.
7. Otherwise collect mismatch rows → fill `OptPrompt` → POST to webhook → use response as next prompt.

In [18]:
os.makedirs(RESULT_DIR, exist_ok=True)

current_prompt = prompt_0
all_rounds: list[dict] = []

for round_idx in range(MAX_ROUNDS):
    round_name  = f"prompt_{round_idx}"
    out_path    = os.path.join(RESULT_DIR, f"test_result_{round_name}.xlsx")
    prompt_path = os.path.join(RESULT_DIR, f"{round_name}.txt")

    print(f"\n{'=' * 60}")
    print(f"Round {round_idx}  |  {round_name}")
    print(f"{'=' * 60}")

    # ── Save prompt as text file ───────────────────────────────────────────
    with open(prompt_path, "w", encoding="utf-8") as fh:
        fh.write(current_prompt)
    print(f"  Saved prompt → {prompt_path}")

    # ── Check if result xlsx already exists — skip evaluation if so ───────
    if os.path.exists(out_path):
        print(f"  Result already exists → {out_path}  (skipping evaluation)")
        df_test_result = pd.read_excel(out_path)
        df_pred = pd.DataFrame({f: df_test_result[f"pred_{f}"].values for f in SCORE_FIELDS})
    else:
        # ── 1. Run evaluation ──────────────────────────────────────────────
        predicted: list[dict] = []
        errors:    list[str | None] = []

        for _, row in tqdm(df_golden.iterrows(), total=len(df_golden), desc=round_name):
            try:
                filled = fill_prompt(current_prompt, row)
                raw    = call_webhook(filled)
                parsed = parse_json_response(raw)
                predicted.append({f: parsed.get(f) for f in SCORE_FIELDS})
                errors.append(None)
            except Exception as exc:
                predicted.append({f: None for f in SCORE_FIELDS})
                errors.append(str(exc))
            time.sleep(DELAY_BETWEEN)

        df_pred = pd.DataFrame(predicted)

        # ── 2. Build test_result dataframe ────────────────────────────────
        df_test_result = df_golden[["testId", "situation_tested", "userMessage"]].copy()
        for f in SCORE_FIELDS:
            df_test_result[golden_col(f)]  = df_golden[golden_col(f)].values
            df_test_result[f"pred_{f}"]    = df_pred[f].values
            df_test_result[f"match_{f}"]   = (df_pred[f].values == df_golden[golden_col(f)].values)
        df_test_result["error"] = errors

        df_test_result.to_excel(out_path, index=False)
        print(f"  Saved → {out_path}")

    # ── 3. Compute per-field mismatch rates ───────────────────────────────
    mismatch_rates: dict[str, float] = {}
    for f in SCORE_FIELDS:
        valid = df_golden[golden_col(f)].notna() & df_pred[f].notna()
        if valid.sum():
            mismatch_rates[f] = float((~df_test_result[f"match_{f}"][valid]).mean())
        else:
            mismatch_rates[f] = float("nan")

    print("\n  Mismatch rates:")
    for f, rate in mismatch_rates.items():
        tag = "✓" if (not pd.isna(rate) and rate <= MISMATCH_THRESHOLD) else "✗"
        print(f"    {f:25s}: {rate:.1%}  {tag}")

    all_rounds.append({
        "round":          round_idx,
        "prompt_name":    round_name,
        "prompt":         current_prompt,
        "df_test_result": df_test_result.copy(),
        "mismatch_rates": mismatch_rates.copy(),
    })

    # ── 4. Stopping condition ─────────────────────────────────────────────
    valid_rates  = [v for v in mismatch_rates.values() if not pd.isna(v)]
    max_mismatch = max(valid_rates) if valid_rates else float("nan")

    if not pd.isna(max_mismatch) and max_mismatch <= MISMATCH_THRESHOLD:
        print(f"\n  All fields within {MISMATCH_THRESHOLD:.0%} threshold. Stopping.")
        break

    if round_idx == MAX_ROUNDS - 1:
        print(f"\n  Reached maximum {MAX_ROUNDS} rounds. Stopping.")
        break

    # ── 5. Collect mismatched rows ─────────────────────────────────────────
    mismatch_mask = pd.Series(False, index=df_golden.index)
    for f in SCORE_FIELDS:
        mismatch_mask |= (df_pred[f].values != df_golden[golden_col(f)].values)

    context_cols      = [
        "testId", "situation_tested", "userMessage", "LastAImessage",
        "narrative", "offerReadableText",
        "result_subject", "result_objective", "result_debt_cause",
        "result_offer_suitability", "result_request_information", "result_summary",
    ]
    golden_label_cols = [golden_col(f) for f in SCORE_FIELDS]

    df_mismatch = (
        df_golden[mismatch_mask][context_cols + golden_label_cols]
        .copy()
        .reset_index(drop=True)
    )
    df_pred_mis = df_pred[mismatch_mask.values].reset_index(drop=True)
    for f in SCORE_FIELDS:
        df_mismatch[f"pred_{f}"] = df_pred_mis[f].values

    mismatch_json = df_mismatch.to_json(orient="records", force_ascii=False, indent=2)
    print(f"\n  {mismatch_mask.sum()} mismatched rows → OptPrompt.")

    # ── 6. Get improved prompt ─────────────────────────────────────────────
    opt_filled = fill_opt_prompt(OptPrompt, current_prompt, mismatch_json)
    print("  Requesting improved prompt…")
    opt_resp       = call_webhook(opt_filled)
    current_prompt = extract_prompt_text(opt_resp)
    print(f"  New prompt ({len(current_prompt)} chars) ready for round {round_idx + 1}.")

print(f"\nOptimisation complete — {len(all_rounds)} round(s) executed.")


Round 0  |  prompt_0
  Saved prompt → summary_test_result/prompt_0.txt


prompt_0:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → summary_test_result/test_result_prompt_0.xlsx

  Mismatch rates:
    subject                  : 13.3%  ✓
    objective                : 40.0%  ✗
    debt_cause               : 16.7%  ✓
    offer_suitability        : 20.0%  ✓
    request_information      : 0.0%  ✓
    summary                  : 16.7%  ✓

  20 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (6396 chars) ready for round 1.

Round 1  |  prompt_1
  Saved prompt → summary_test_result/prompt_1.txt


prompt_1:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → summary_test_result/test_result_prompt_1.xlsx

  Mismatch rates:
    subject                  : 46.7%  ✗
    objective                : 66.7%  ✗
    debt_cause               : 20.0%  ✓
    offer_suitability        : 33.3%  ✗
    request_information      : 0.0%  ✓
    summary                  : 50.0%  ✗

  26 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (6740 chars) ready for round 2.

Round 2  |  prompt_2
  Saved prompt → summary_test_result/prompt_2.txt


...

  Saved → summary_test_result/test_result_prompt_2.xlsx

  Mismatch rates:
    subject                  : 13.3%  ✓
    objective                : 33.3%  ✗
    debt_cause               : 26.7%  ✓
    offer_suitability        : 26.7%  ✓
    request_information      : 0.0%  ✓
    summary                  : 30.0%  ✓

  24 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (6753 chars) ready for round 3.

Round 3  |  prompt_3
  Saved prompt → summary_test_result/prompt_3.txt


prompt_3:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → summary_test_result/test_result_prompt_3.xlsx

  Mismatch rates:
    subject                  : 30.0%  ✓
    objective                : 50.0%  ✗
    debt_cause               : 20.0%  ✓
    offer_suitability        : 26.7%  ✓
    request_information      : 0.0%  ✓
    summary                  : 60.0%  ✗

  27 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (7227 chars) ready for round 4.

Round 4  |  prompt_4
  Saved prompt → summary_test_result/prompt_4.txt


prompt_4:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → summary_test_result/test_result_prompt_4.xlsx

  Mismatch rates:
    subject                  : 26.7%  ✓
    objective                : 50.0%  ✗
    debt_cause               : 26.7%  ✓
    offer_suitability        : 16.7%  ✓
    request_information      : 0.0%  ✓
    summary                  : 20.0%  ✓

  Reached maximum 5 rounds. Stopping.

Optimisation complete — 5 round(s) executed.


## Results Summary

In [19]:
summary_rows = []
for r in all_rounds:
    row = {"round": r["round"], "prompt_name": r["prompt_name"]}
    row.update({f: f'{v:.1%}' for f, v in r["mismatch_rates"].items()})
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("=" * 60)
print("OPTIMISATION SUMMARY — mismatch rate per field")
print("=" * 60)
display(df_summary)

best = min(
    all_rounds,
    key=lambda r: max(
        (v for v in r["mismatch_rates"].values() if not pd.isna(v)),
        default=float("inf"),
    ),
)
print(f"\nBest round : {best['prompt_name']}")
print(
    f"Max mismatch: "
    f"{max(v for v in best['mismatch_rates'].values() if not pd.isna(v)):.1%}"
)
print("\nAccess results:")
print("  all_rounds[N]['df_test_result']  — labelled DataFrame for round N")
print("  all_rounds[N]['prompt']          — prompt text used in round N")
print("  all_rounds[-1]['prompt']         — final (best last) prompt")

OPTIMISATION SUMMARY — mismatch rate per field


,round,prompt_name,subject,objective,debt_cause,offer_suitability,request_information,summary
0,0,prompt_0,13.3%,40.0%,16.7%,20.0%,0.0%,16.7%
1,1,prompt_1,46.7%,66.7%,20.0%,33.3%,0.0%,50.0%
2,2,prompt_2,13.3%,33.3%,26.7%,26.7%,0.0%,30.0%
3,3,prompt_3,30.0%,50.0%,20.0%,26.7%,0.0%,60.0%
4,4,prompt_4,26.7%,50.0%,26.7%,16.7%,0.0%,20.0%



Best round : prompt_2
Max mismatch: 33.3%

Access results:
  all_rounds[N]['df_test_result']  — labelled DataFrame for round N
  all_rounds[N]['prompt']          — prompt text used in round N
  all_rounds[-1]['prompt']         — final (best last) prompt
